In [ ]:
# Cell 1
# versions mirror ../../PINS.md; if a pin changes, change it there first
!pip -q install "transformers==4.46.*" "accelerate==1.1.*" "bitsandbytes==0.49.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 34.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
# Cell 2
import torch, subprocess

assert torch.cuda.is_available(), "no GPU: set Runtime > Change runtime type > T4 GPU"

gpu_name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print("gpu:", gpu_name)
print("total VRAM: %.2f GB" % total_gb)

# nvidia-smi is the ground truth the whole course reads.
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip())

gpu: Tesla T4
total VRAM: 15.64 GB
3 MiB, 15360 MiB


In [ ]:
# Cell 3
import torch, time, json, gc

def measured_vram_gb():
    """Bytes CUDA is holding for this process, in GB. This is what fills the card."""
    torch.cuda.synchronize()
    return torch.cuda.memory_reserved(0) / 1e9

def free_vram():
    """Hand freed memory back to the driver.

    Delete the model variable yourself first, in the cell, with `del model_x`.
    Python frees an object when the last reference to it goes away, and your
    notebook variable is a reference. No helper can delete a name that lives in
    your cell, so this function only does the second half of the job.
    """
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(0)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print("helper ready for", MODEL_ID)

helper ready for Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
# Cell 4
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)

before = measured_vram_gb()
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="cuda")
after = measured_vram_gb()

fp16_measured = after
print("fp16 resident VRAM: %.2f GB" % fp16_measured)
print("delta from before-load: %.2f GB" % (after - before))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

fp16 resident VRAM: 3.29 GB
delta from before-load: 3.29 GB


In [ ]:
# Cell 5
from transformers import BitsAndBytesConfig

del model_fp16   # your reference to the weights; without this line nothing frees
free_vram()

model_int8 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map="cuda")
int8_measured = measured_vram_gb()
print("int8 resident VRAM: %.2f GB" % int8_measured)

int8 resident VRAM: 1.87 GB


In [ ]:
# Cell 6
del model_int8
free_vram()

model_int4 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True),
    device_map="cuda")
int4_measured = measured_vram_gb()
print("int4 resident VRAM: %.2f GB" % int4_measured)

int4 resident VRAM: 1.24 GB


In [ ]:
# Cell 7
generate_src = '''
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

def load(dtype):
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    if dtype == "fp16":
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="cuda")
    elif dtype == "int8":
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map="cuda")
    elif dtype == "int4":
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=BitsAndBytesConfig(load_in_4bit=True), device_map="cuda")
    else:
        raise ValueError(dtype)
    return tok, m

def tokens_per_s(dtype, new_tokens=128):
    tok, m = load(dtype)
    msgs = [{"role": "user", "content": "Explain what a GPU does, in three sentences."}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    m.generate(**{"input_ids": ids}, max_new_tokens=8)  # warm-up, not timed
    torch.cuda.synchronize()
    t0 = time.time()
    out = m.generate(**{"input_ids": ids}, max_new_tokens=new_tokens, do_sample=False)
    torch.cuda.synchronize()
    dt = time.time() - t0
    generated = out.shape[1] - ids.shape[1]
    return generated / dt

if __name__ == "__main__":
    for d in ["fp16", "int8", "int4"]:
        print(d, "%.1f tok/s" % tokens_per_s(d))
'''

with open("generate.py", "w") as f:
    f.write(generate_src)
print("wrote generate.py")

wrote generate.py


In [ ]:
# Cell 8
import importlib.util
spec = importlib.util.spec_from_file_location("gen", "generate.py")
gen = importlib.util.module_from_spec(spec)

# free the int4 model from Cell 6 so the script loads cleanly
del model_int4
free_vram()
spec.loader.exec_module(gen)

# Inside tokens_per_s the model is a local variable, so it frees itself when the
# function returns. Here free_vram() alone is enough; there is no name to delete.
fp16_tps = gen.tokens_per_s("fp16"); print("fp16 %.1f tok/s" % fp16_tps)
free_vram()
int8_tps = gen.tokens_per_s("int8"); print("int8 %.1f tok/s" % int8_tps)
free_vram()
int4_tps = gen.tokens_per_s("int4"); print("int4 %.1f tok/s" % int4_tps)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20`

fp16 15.3 tok/s
int8 5.5 tok/s


/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


int4 13.9 tok/s


In [ ]:
# Cell 9
free_vram()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="cuda")
base = measured_vram_gb()
print("weights only: %.2f GB" % base)

for ctx in [256, 1024, 3072]:
    prompt = "word " * ctx
    ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")
    m.generate(**{"input_ids": ids}, max_new_tokens=64, do_sample=False)
    peak = torch.cuda.max_memory_reserved(0) / 1e9
    print("ctx ~%d tokens -> peak VRAM %.2f GB (KV + activations: %.2f GB)" % (ctx, peak, peak - base))
    torch.cuda.reset_peak_memory_stats(0)

weights only: 3.31 GB
ctx ~256 tokens -> peak VRAM 3.34 GB (KV + activations: 0.03 GB)
ctx ~1024 tokens -> peak VRAM 3.44 GB (KV + activations: 0.13 GB)
ctx ~3072 tokens -> peak VRAM 3.65 GB (KV + activations: 0.34 GB)


In [ ]:
# Cell 10
import json

results = {
    "model": MODEL_ID,
    "gpu": gpu_name,
    "measurements": [
        {"dtype": "fp16", "predicted_gb": None, "measured_gb": round(fp16_measured, 2), "tokens_per_s": round(fp16_tps, 1)},
        {"dtype": "int8", "predicted_gb": None, "measured_gb": round(int8_measured, 2), "tokens_per_s": round(int8_tps, 1)},
        {"dtype": "int4", "predicted_gb": None, "measured_gb": round(int4_measured, 2), "tokens_per_s": round(int4_tps, 1)},
    ],
}

# fill predicted_gb from your prediction card (weights only is fine):
results["measurements"][0]["predicted_gb"] = 3.0   # <- your fp16 estimate
results["measurements"][1]["predicted_gb"] = 1.5   # <- your int8 estimate
results["measurements"][2]["predicted_gb"] = 0.75  # <- your int4 estimate

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "gpu": "Tesla T4",
  "measurements": [
    {
      "dtype": "fp16",
      "predicted_gb": 3.0,
      "measured_gb": 3.29,
      "tokens_per_s": 15.3
    },
    {
      "dtype": "int8",
      "predicted_gb": 1.5,
      "measured_gb": 1.87,
      "tokens_per_s": 5.5
    },
    {
      "dtype": "int4",
      "predicted_gb": 0.75,
      "measured_gb": 1.24,
      "tokens_per_s": 13.9
    }
  ]
}


In [ ]:
# Paste this whole cell into a fresh Colab cell after you have written
# results.json and generate.py into the working directory.
# It prints exactly one line last: GREEN CHECK: PASS  or  GREEN CHECK: FAIL (<reason>)
# stdlib only. No arguments, no interactivity.
import json, os


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def _fail(reason):
    print("GREEN CHECK: FAIL (%s)" % reason)
    raise _Stop()


def main():
    # 1. the generation script must exist on disk
    if not os.path.isfile("generate.py"):
        _fail("generate.py not found next to this cell")

    # 2. results.json must exist and parse
    if not os.path.isfile("results.json"):
        _fail("results.json not found; run the results cell first")
    try:
        with open("results.json") as f:
            data = json.load(f)
    except json.JSONDecodeError as e:
        _fail("results.json is not valid JSON: %s" % e)

    # 3. top-level schema
    for key in ("model", "gpu", "measurements"):
        if key not in data:
            _fail("results.json missing top-level key '%s'" % key)
    rows = data["measurements"]
    if not isinstance(rows, list) or len(rows) != 3:
        _fail("measurements must be a list of 3 rows (fp16, int8, int4)")

    by_dtype = {}
    for row in rows:
        for field in ("dtype", "predicted_gb", "measured_gb", "tokens_per_s"):
            if field not in row:
                _fail("a measurement row is missing '%s'" % field)
        dt = row["dtype"]
        if dt not in ("fp16", "int8", "int4"):
            _fail("unexpected dtype '%s'" % dt)
        for field in ("predicted_gb", "measured_gb", "tokens_per_s"):
            v = row[field]
            if not isinstance(v, (int, float)):
                _fail("%s.%s is not a number (fill it in)" % (dt, field))
            if v <= 0:
                _fail("%s.%s must be positive, got %s" % (dt, field, v))
        by_dtype[dt] = row

    for dt in ("fp16", "int8", "int4"):
        if dt not in by_dtype:
            _fail("missing the %s row" % dt)

    fp16 = by_dtype["fp16"]["measured_gb"]
    int8 = by_dtype["int8"]["measured_gb"]
    int4 = by_dtype["int4"]["measured_gb"]

    # 4. ranges sane: fp16 weights+overhead land in a physical window on a T4
    if not (2.5 <= fp16 <= 6.0):
        _fail("fp16 measured %.2f GB outside sane 2.5-6.0 GB; remeasure" % fp16)

    # 5. ordering on memory: int4 < int8 < fp16
    if not (int4 < int8 < fp16):
        _fail("memory order wrong: expected int4 < int8 < fp16, got %.2f, %.2f, %.2f"
              % (int4, int8, fp16))

    print("model:", data["model"])
    print("gpu:  ", data["gpu"])
    print("fp16 %.2f GB | int8 %.2f GB | int4 %.2f GB" % (fp16, int8, int4))
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)

model: Qwen/Qwen2.5-1.5B-Instruct
gpu:   Tesla T4
fp16 3.29 GB | int8 1.87 GB | int4 1.24 GB
GREEN CHECK: PASS


In [ ]:
# Cell S (stretch)
del m            # the fp16 model Cell 9 left resident
free_vram()
# fp32 is 4 bytes per parameter: 1.5B x 4 = ~6 GB weights, plus a fat context.
m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32, device_map="cuda")
huge = "word " * 20000
ids = tok(huge, return_tensors="pt").input_ids.to("cuda")
m.generate(**{"input_ids": ids}, max_new_tokens=2000)  # expect CUDA out of memory

KeyboardInterrupt: 